# **IMDB Word2Vec: Pipeline 0**

This experiment implements a classical NLP baseline using the
**Bag of Words (BoW)** representation over the raw IMDB dataset
without preprocessing transformations.

Pipeline 0 preserves the original textual structure of the reviews,
allowing evaluation of how traditional NLP models behave without
token normalization or linguistic cleaning.

The objective of this experiment is to:

- Build Bag of Words representations
- Train baseline machine learning classifiers
- Evaluate classification performance
- Compare traditional NLP approaches against future embedding architectures



# **1. Import Libraries**

In [2]:
import pandas as pd
import numpy as np
import os
import time

from transformers import BertTokenizer, BertModel
import torch

from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.svm import LinearSVC

c:\Users\Ale\Documents\Final_NLP\venv_gpu\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
#import sys
#!{sys.executable} -m pip install transformers

# **2. Load Dataset**

Pipeline 0 uses the raw IMDB dataset without preprocessing.

The dataset is divided into:
- Training set
- Validation set
- Testing set

In [4]:
train = pd.read_csv(
    r"C:\Users\Ale\Documents\Final_NLP\Homework_2\Text-Representation-and-Basic-Text-Mining\Text_Classification_and_Embeddings\Models\Data\preprocessed_data\pipeline_0\train.csv"
)

val = pd.read_csv(
    r"C:\Users\Ale\Documents\Final_NLP\Homework_2\Text-Representation-and-Basic-Text-Mining\Text_Classification_and_Embeddings\Models\Data\preprocessed_data\pipeline_0\validation.csv"
)

test = pd.read_csv(
    r"C:\Users\Ale\Documents\Final_NLP\Homework_2\Text-Representation-and-Basic-Text-Mining\Text_Classification_and_Embeddings\Models\Data\preprocessed_data\pipeline_0\test.csv"
)

print("Train Shape:", train.shape)
print("Validation Shape:", val.shape)
print("Test Shape:", test.shape)

Train Shape: (20000, 2)
Validation Shape: (2500, 2)
Test Shape: (2500, 2)


# **3. BERT Embedding Extraction Function**


This section implements transformer-based contextual embeddings using
a pre-trained BERT model.

Unlike Word2Vec, BERT generates contextual representations where the
embedding of a word depends on the surrounding sentence context.

The objective of this implementation is to:

- Extract contextual sentence embeddings
- Use the CLS token representation
- Generate dense semantic vectors
- Train classifiers on top of transformer embeddings

This implementation follows the feature extraction approach:

```text
Text
→ BERT Tokenizer
→ BERT Model
→ CLS Embedding
→ Dense Sentence Vector
→ Classifier
```

The transformer model remains frozen during embedding extraction.

In [ ]:
def compute_bert_embeddings(
    X_train,
    X_val,
    X_test,
    model_name="bert-base-uncased",
    max_length=256,
    batch_size=16
):

    """
    Computes BERT sentence embeddings
    using CLS token extraction.

    Steps:
    1. Load pretrained BERT tokenizer
    2. Load pretrained BERT model
    3. Tokenize text
    4. Extract CLS embeddings
    5. Generate sentence embeddings

    Returns:
    - X_train_bert
    - X_val_bert
    - X_test_bert
    - tokenizer
    - bert_model
    """

    # Load tokenizer
    tokenizer = BertTokenizer.from_pretrained(
        model_name
    )

    # Load pretrained BERT model
    bert_model = BertModel.from_pretrained(
        model_name
    )

    # Evaluation mode
    bert_model.eval()

    # Device configuration
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    bert_model.to(device)
    print(f"Using device: {device}")

    # Embedding Extraction Function
    def extract_embeddings(texts):
        embeddings = []
        # Disable gradients
        with torch.no_grad():

            # Process texts in batches
            for i in range(0, len(texts), batch_size):

                batch_texts = texts[i:i + batch_size]

                # Tokenization
                encoded = tokenizer(
                    batch_texts.tolist(),
                    padding=True,
                    truncation=True,
                    max_length=max_length,
                    return_tensors="pt"
                )

                # Move tensors to device
                input_ids = encoded['input_ids'].to(device)
                attention_mask = encoded['attention_mask'].to(device)

                # Forward pass
                outputs = bert_model(
                    input_ids=input_ids,
                    attention_mask=attention_mask
                )

                # CLS Token Extraction
                # Shape: (batch_size, hidden_size)
                # hidden_size = 768 for BERT Base
                cls_embeddings = outputs.last_hidden_state[:, 0, :]
                # Move to CPU
                cls_embeddings = cls_embeddings.cpu().numpy()
                embeddings.extend(cls_embeddings)
        return np.array(embeddings)

    # -----------------------------------------
    # Generate embeddings
    # -----------------------------------------

    print("Generating BERT embeddings for training set...")
    X_train_bert = extract_embeddings(X_train)

    print("Generating BERT embeddings for validation set...")
    X_val_bert = extract_embeddings(X_val)

    print("Generating BERT embeddings for testing set...")
    X_test_bert = extract_embeddings(X_test)

    return (
        X_train_bert,
        X_val_bert,
        X_test_bert,
        tokenizer,
        bert_model
    )

# **4. Prepare Features and Labels**

In [6]:
X_train = train['text']
y_train = train['label']

X_val = val['text']
y_val = val['label']

X_test = test['text']
y_test = test['label']

In [ ]:
X_train_bert, X_val_bert, X_test_bert, bert_tokenizer, bert_model = compute_bert_embeddings(
    X_train,
    X_val,
    X_test
)

print("Train Shape:", X_train_bert.shape)
print("Validation Shape:", X_val_bert.shape)
print("Test Shape:", X_test_bert.shape)

c:\Users\Ale\Documents\Final_NLP\venv_gpu\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Ale\.cache\huggingface\hub\models--bert-base-uncased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 8473.17it/s]
[transformers] BertModel LOAD REPORT from: 

Using device: cuda
Generating BERT embeddings for training set...


# **5. Model Training**

Two classical machine learning classifiers are trained:

- Logistic Regression
- Multinomial Naive Bayes

These models are commonly used as traditional NLP baselines.

In [ ]:
def train_model(model, X_train, y_train):

    """
    Trains a machine learning model.
    """
    model.fit(X_train, y_train)
    return model

In [ ]:
# Logistic Regression
logistic_model = LogisticRegression(max_iter=1000)

logistic_model = train_model(
    logistic_model,
    X_train_bert,
    y_train
)

In [ ]:
# SuperVector Machine
svm_model = LinearSVC()

svm_model = train_model(
    svm_model,
    X_train_bert,
    y_train
)

# **6. Evaluation Metrics**

The following metrics are computed:

- Accuracy
- Precision
- Recall
- F1-Score

These metrics evaluate classification performance
from different perspectives.

In [ ]:

def compute_metrics(model, X_test, y_test):

    """
    Computes classification metrics
    and inference time.
    """

    start_time = time.time()
    # Predictions
    predictions = model.predict(X_test)
    end_time = time.time()
    inference_time = end_time - start_time

    # Average inference time per sample
    avg_inference_time = inference_time / len(y_test)

    # Compute metrics
    metrics = {
        "Accuracy": accuracy_score(y_test, predictions),
        "Precision": precision_score(y_test, predictions),
        "Recall": recall_score(y_test, predictions),
        "F1-Score": f1_score(y_test, predictions),
        "Inference_Time_Seconds": inference_time,
        "Average_Inference_Time_Per_Sample":avg_inference_time      
    }

    return metrics, predictions

In [ ]:
# Logistic Regression Metrics
log_metrics, log_predictions = compute_metrics(
    logistic_model,
    X_test_bert,
    y_test
)

print("Logistic Regression Metrics")
print(log_metrics)

Logistic Regression Metrics
{'Accuracy': 0.83, 'Precision': 0.8245476003147129, 'Recall': 0.8384, 'F1-Score': 0.8314161047203491, 'Inference_Time_Seconds': 0.0023145675659179688, 'Average_Inference_Time_Per_Sample': 9.258270263671875e-07}


In [ ]:
# SVM Metrics
svm_metrics, svm_predictions = compute_metrics(
    svm_model,
    X_test_bert,
    y_test
)

print("\nSVM Metrics")
print(svm_metrics)


SVM Metrics
{'Accuracy': 0.8388, 'Precision': 0.8326787117046347, 'Recall': 0.848, 'F1-Score': 0.8402695204122077, 'Inference_Time_Seconds': 0.002597808837890625, 'Average_Inference_Time_Per_Sample': 1.03912353515625e-06}


# **7. Confusion Matrix**

The confusion matrix allows visualization of:

- True Positives
- True Negatives
- False Positives
- False Negatives

This helps analyze classification behavior and prediction errors.

In [ ]:
def plot_confusion_matrix(
    y_true,
    y_pred,
    model_name,
    pipeline_name,
    representation_name,
    output_dir="results_w2v"
):

    """
    Computes and saves confusion matrix.

    The confusion matrix includes:
    - Model name
    - Pipeline name
    - Representation type

    Confusion matrices are saved inside:
    conf_matrix/
    """

    conf_matrix_dir = os.path.join(
        output_dir,
        "conf_matrix"
    )

    os.makedirs(conf_matrix_dir, exist_ok=True)

    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(6, 5))

    sns.heatmap(
        cm,
        annot=True,
        fmt='d',
        cmap='Blues'
    )

    plt.title(
        f"{model_name} | {pipeline_name} | {representation_name}"
    )

    plt.xlabel("Predicted")
    plt.ylabel("Actual")

    file_name = (
        f"{model_name}_"
        f"{pipeline_name}_"
        f"{representation_name}_"
        f"confusion_matrix.png"
    )

    save_path = os.path.join(
        conf_matrix_dir,
        file_name
    )

    plt.savefig(
        save_path,
        bbox_inches='tight'
    )

    plt.close()

    print(f"Confusion matrix saved in:\n{save_path}")

In [ ]:
# Create Results Directory
results_dir = "results_w2v"
os.makedirs(results_dir, exist_ok=True)


# Plot Confusion Matrices

plot_confusion_matrix(
    y_true=y_test,
    y_pred=log_predictions,
    model_name="LogisticRegression",
    pipeline_name="pipeline_0",
    representation_name="Word2Vec"
)

plot_confusion_matrix(
    y_true=y_test,
    y_pred=svm_predictions,
    model_name="SVM",
    pipeline_name="pipeline_0",
    representation_name="Word2Vec"
)

Confusion matrix saved in:
results_w2v\conf_matrix\LogisticRegression_pipeline_0_Word2Vec_confusion_matrix.png
Confusion matrix saved in:
results_w2v\conf_matrix\SVM_pipeline_0_Word2Vec_confusion_matrix.png


# **8. Save Results**

All evaluation metrics and reports are saved
inside the `results_bag/` directory for future analysis.

In [ ]:
def save_results(
    metrics,
    model_name,
    pipeline_name,
    representation_name,
    output_dir="results_w2v",
    results_file="experiment_results.csv"
):

    """
    Saves experiment results into a single CSV file.
    Each experiment is appended as a new row.
    """

    # Create results directory
    os.makedirs(output_dir, exist_ok=True)

    # Full CSV path
    save_path = os.path.join(
        output_dir,
        results_file
    )

    # Create experiment dictionary
    results = {
        "Model": model_name,
        "Pipeline": pipeline_name,
        "Representation": representation_name,
        **metrics
    }

    # Convert current experiment to dataframe
    current_result_df = pd.DataFrame([results])

    # Append if file exists
    if os.path.exists(save_path):
        existing_results = pd.read_csv(save_path)
        updated_results = pd.concat(
            [existing_results, current_result_df],
            ignore_index=True
        )
    else:
        updated_results = current_result_df

    # Save updated dataframe
    updated_results.to_csv(
        save_path,
        index=False
    )

    print(f"Results appended to:\n{save_path}")

In [ ]:
# Save Logistic Regression Results
save_results(
    metrics=log_metrics,
    model_name="LogisticRegression",
    pipeline_name="pipeline_0",
    representation_name="Bert"
)

# Save SVM Results
save_results(
    metrics=svm_metrics,
    model_name="SVM",
    pipeline_name="pipeline_0",
    representation_name="Bert"
)
    

print("Results saved successfully inside:")
print(results_dir)

Results appended to:
results_w2v\experiment_results.csv
Results appended to:
results_w2v\experiment_results.csv
Results saved successfully inside:
results_w2v
